# Onboarding 01 ? Data preparation

This notebook introduces HEST metadata and the breast spatial-transcriptomics cohort, previews one paired H&E patch and expression profile, then prepares aligned files for modeling. The catalog has one row per tissue section; sections belong to patients and contain many measured spots. Expression is stored as a sparse spot-by-gene matrix.

The test run selects three patients; set `N_PATIENTS = None` for all 23. Set `DATA_DIR` to the shared Drive location, a local directory, or a Google Drive Desktop mirrored directory. Keep the same root in Notebooks 02?03. Detailed concept notes: [study materials](../../docs/study_materials.md).

**Creates:** raw patches/expression, `spot_table.csv`, `samples.csv`, `genes.txt`, and sparse log1p expression matrices.


In [ ]:
%pip install -q huggingface_hub anndata h5py scipy pandas matplotlib


In [ ]:
import os
import numpy as np
import pandas as pd
import h5py
import anndata
import scipy.sparse
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download, login

**Storage:** Mount Drive in Colab only if the data root is on Drive. Local and Drive Desktop paths work by setting `DATA_DIR` in Settings.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Local runtime: no Drive mount needed.')

## Settings

Everything you might change is in this cell. Nothing below it should need editing.

- `TEAM_FOLDER`: the shared team folder on Drive
- `DATA_DIR`: where this notebook saves its data, inside the team folder
- `N_PATIENTS`: how many patients to use. `3` for the test run, `None` for all 23
- `STUDY_TITLE_START`: which study to use (the start of its `dataset_title` in the HEST table)

In [ ]:
from pathlib import Path
# Examples: '/content/drive/MyDrive/BDD_BJMJ/HEST_breast_ST', './data/HEST_breast_ST', or your Drive Desktop folder.
DATA_DIR = os.environ.get('HIST2ST_DATA_DIR', './data/HEST_breast_ST')
METADATA_FILE = 'HEST_v1_1_0.csv'
ORGAN, SPECIES, TECHNOLOGY = 'Breast', 'Homo sapiens', 'Spatial Transcriptomics'
STUDY_TITLE_START = 'Integrating spatial gene expression and breast tumour morphology'
N_PATIENTS = 3  # Set to None for all 23 patients.


In [ ]:
DATA_DIR = str(Path(DATA_DIR).expanduser().resolve())
RAW_DIR = os.path.join(DATA_DIR, 'raw')
EXPRESSION_DIR = os.path.join(DATA_DIR, 'expression')
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(EXPRESSION_DIR, exist_ok=True)
print('Saving to:', DATA_DIR)


**Hugging Face login:** paste your token into the box that appears. Never write it in the code or upload it to GitHub.

In [ ]:
login()

## Step 1. Choose the sections

The HEST table has one row per section. We keep the human breast sections from our study, and drop any without a patient ID (this study has none missing, but other studies do).

In this study the patient IDs look like `BC23209`, and each patient has 2–3 sections. Several sections from one patient are why the patient split matters: they must all land on the same side of the split.

In [ ]:
metadata_path = hf_hub_download(
    repo_id="MahmoodLab/hest",
    repo_type="dataset",
    filename=METADATA_FILE,
    local_dir=RAW_DIR,
)
metadata = pd.read_csv(metadata_path, encoding="utf-8-sig")
print("All samples in HEST:", len(metadata))

is_organ = metadata["organ"] == ORGAN
is_species = metadata["species"] == SPECIES
is_technology = metadata["st_technology"] == TECHNOLOGY
is_study = metadata["dataset_title"].str.startswith(STUDY_TITLE_START)
has_patient = metadata["patient"].notna()

study_samples = metadata[is_organ & is_species & is_technology & is_study & has_patient].copy()

print("Sections in our study:", len(study_samples))
print("Patients in our study:", study_samples["patient"].nunique())

### Inspect the metadata catalog
Each row is a section/sample. These summaries show the shape of the catalog, its columns, available technologies, and missing patient IDs.

In [ ]:
print('Metadata shape:', metadata.shape)
print('Columns:', metadata.columns.tolist())
print('Missing patient IDs:', int(metadata['patient'].isna().sum()))
human = metadata[metadata['species'].eq('Homo sapiens')]
display(human['organ'].value_counts().head(10).rename('sections').to_frame())
display(human['st_technology'].value_counts().rename('sections').to_frame())


In [ ]:
sections_per_patient = study_samples.groupby("patient")["id"].apply(list)
print(sections_per_patient.to_string())

For the test run we keep the first `N_PATIENTS` patients (sorted by ID, so everyone gets the same ones) with **all** of their sections.

In [ ]:
all_patients = sorted(study_samples["patient"].unique())

if N_PATIENTS is None:
    chosen_patients = all_patients
else:
    chosen_patients = all_patients[:N_PATIENTS]

samples = study_samples[study_samples["patient"].isin(chosen_patients)]
samples = samples.sort_values(["patient", "id"]).reset_index(drop=True)
sample_ids = samples["id"].tolist()

print("Patients used:", len(chosen_patients), chosen_patients)
print("Sections used:", len(sample_ids), sample_ids)
print("Spots expected (from metadata):", samples["spots_under_tissue"].sum())

## Step 2. Download patches and expression

For each section we download two files:
- `patches/<id>.h5`: one 224×224 H&E patch per spot, plus its barcode and coordinates
- `st/<id>.h5ad`: the expression table, spots × genes, with each spot's barcode

Files that are already downloaded are skipped, so if the run stops (or Colab disconnects) you can just run this cell again.

In [ ]:
for sample_id in sample_ids:
    patches_path = os.path.join(RAW_DIR, "patches", f"{sample_id}.h5")
    st_path = os.path.join(RAW_DIR, "st", f"{sample_id}.h5ad")

    if os.path.exists(patches_path) and os.path.exists(st_path):
        print(sample_id, "already downloaded")
        continue

    hf_hub_download(repo_id="MahmoodLab/hest", repo_type="dataset",
                    filename=f"patches/{sample_id}.h5", local_dir=RAW_DIR)
    hf_hub_download(repo_id="MahmoodLab/hest", repo_type="dataset",
                    filename=f"st/{sample_id}.h5ad", local_dir=RAW_DIR)
    print(sample_id, "downloaded")

In [ ]:
total_bytes = 0
for sample_id in sample_ids:
    total_bytes += os.path.getsize(os.path.join(RAW_DIR, "patches", f"{sample_id}.h5"))
    total_bytes += os.path.getsize(os.path.join(RAW_DIR, "st", f"{sample_id}.h5ad"))

size_per_section = total_bytes / len(sample_ids)
print(f"Downloaded: {total_bytes / 1e9:.2f} GB for {len(sample_ids)} sections")
print(f"Estimate for all {len(study_samples)} sections: {size_per_section * len(study_samples) / 1e9:.1f} GB")

## Step 3. Match patches to expression by barcode

The barcode is the spot's ID. A patch and an expression row belong to the same spot only if their barcodes match, so we never line them up by position.

For each section we keep the spots that have **both** a patch and an expression row. We also record:
- `patch_index`: where the spot's patch sits in the `.h5` file (Notebook 2 reads patches in this order)
- `row_in_sample`: the spot's row in this section's saved expression file

In [ ]:
def barcode_to_text(barcode):
    if hasattr(barcode, "ravel"):
        barcode = barcode.ravel()[0]
    if isinstance(barcode, bytes):
        barcode = barcode.decode("utf-8")
    return str(barcode)


def read_patch_barcodes(patches_path):
    with h5py.File(patches_path, "r") as f:
        raw_barcodes = f["barcode"][:]
        coords = f["coords"][:]
    barcodes = np.array([barcode_to_text(b) for b in raw_barcodes])
    return barcodes, coords

In [ ]:
spot_tables = []
gene_lists = []

for sample_id in sample_ids:
    patches_path = os.path.join(RAW_DIR, "patches", f"{sample_id}.h5")
    st_path = os.path.join(RAW_DIR, "st", f"{sample_id}.h5ad")

    patch_barcodes, coords = read_patch_barcodes(patches_path)

    adata = anndata.read_h5ad(st_path, backed="r")
    expression_barcodes = set(adata.obs_names)
    gene_names = pd.Index(adata.var_names)
    gene_lists.append(list(anndata.utils.make_index_unique(gene_names)))
    adata.file.close()

    keep = np.array([b in expression_barcodes for b in patch_barcodes])
    patient = samples.loc[samples["id"] == sample_id, "patient"].iloc[0]

    table = pd.DataFrame({
        "sample_id": sample_id,
        "patient": patient,
        "barcode": patch_barcodes[keep],
        "x": coords[keep, 0],
        "y": coords[keep, 1],
        "patch_index": np.where(keep)[0],
    })
    table["row_in_sample"] = np.arange(len(table))
    spot_tables.append(table)

    print(f"{sample_id}: {len(patch_barcodes)} patches, {keep.sum()} matched to expression")

spot_table = pd.concat(spot_tables, ignore_index=True)
print()
print("Total spots:", len(spot_table))

### Preview a paired patch and expression profile
The image patch and expression row are joined by barcode. The plot demonstrates what ?paired data? means: image and gene values describe the same location.

In [ ]:
sample_id = sample_ids[0]
spot = spot_table[spot_table['sample_id'].eq(sample_id)].iloc[0]
with h5py.File(os.path.join(RAW_DIR, 'patches', f'{sample_id}.h5'), 'r') as f:
    patch_image = f['img'][int(spot.patch_index)]
adata_preview = anndata.read_h5ad(os.path.join(RAW_DIR, 'st', f'{sample_id}.h5ad'))
adata_preview.var_names_make_unique()
row = int(adata_preview.obs_names.get_loc(spot.barcode))
values = adata_preview.X[row]
values = values.toarray().ravel() if scipy.sparse.issparse(values) else np.asarray(values).ravel()
top = np.argsort(values)[-8:][::-1]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].imshow(patch_image); axes[0].set_title(f'{sample_id}: {spot.barcode}'); axes[0].axis('off')
axes[1].barh(adata_preview.var_names[top][::-1], values[top][::-1]); axes[1].set_title('Top raw expression values')
plt.tight_layout(); plt.show()
adata_preview.file.close()


## Step 4. Keep the genes every section measured

Different sections can list slightly different genes. Notebook 3 needs the same columns for every spot, so we keep only genes present in **all** chosen sections.

This is not gene selection: it doesn't look at expression values, only at which genes exist, so it can be done before the split. Choosing the top 50 genes by variance happens in Notebook 3, on training spots only.

In [ ]:
shared_genes = set(gene_lists[0])
for genes in gene_lists[1:]:
    shared_genes = shared_genes & set(genes)

shared_genes = sorted(shared_genes)

print("Genes per section:", [len(g) for g in gene_lists])
print("Genes shared by all sections:", len(shared_genes))

## Step 5. Save log1p expression for each section

For each section we:
1. reorder its expression rows to match the spot table (by barcode)
2. keep the shared genes, in the same order for every section
3. apply log1p, so ln(1 + count): this evens out very large counts and keeps zeros at 0
4. save it as a sparse matrix (most counts are 0, so this keeps the files small)

log1p is applied to each number on its own and learns nothing from other spots, so doing it here can't leak test information.

In [ ]:
for sample_id in sample_ids:
    st_path = os.path.join(RAW_DIR, "st", f"{sample_id}.h5ad")
    adata = anndata.read_h5ad(st_path)
    adata.var_names_make_unique()

    sample_spots = spot_table[spot_table["sample_id"] == sample_id]
    adata = adata[sample_spots["barcode"].values, shared_genes]

    counts = scipy.sparse.csr_matrix(adata.X, dtype=np.float32)
    log_expression = counts.log1p()

    scipy.sparse.save_npz(os.path.join(EXPRESSION_DIR, f"{sample_id}.npz"), log_expression)
    print(f"{sample_id}: saved {log_expression.shape[0]} spots x {log_expression.shape[1]} genes")

### Summarize average expression
The saved values are `log1p(count)`. The chart shows average log expression per shared gene and average expression per spot; these are descriptive summaries, not gene selection.

In [ ]:
combined_expression = scipy.sparse.vstack([scipy.sparse.load_npz(os.path.join(EXPRESSION_DIR, f'{sid}.npz')) for sid in sample_ids]).tocsr()
mean_gene = np.asarray(combined_expression.mean(axis=0)).ravel()
mean_spot = np.asarray(combined_expression.mean(axis=1)).ravel()
print('Matrix (spots ? shared genes):', combined_expression.shape)
print('Average log1p value across all entries:', float(combined_expression.mean()))
idx = np.argsort(mean_gene)[-12:][::-1]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(np.asarray(shared_genes)[idx][::-1], mean_gene[idx][::-1]); axes[0].set_title('Highest mean log1p genes')
axes[1].hist(mean_spot, bins=40); axes[1].set_title('Mean log1p value per spot')
plt.tight_layout(); plt.show()


In [ ]:
spot_table.to_csv(os.path.join(DATA_DIR, "spot_table.csv"), index=False)

with open(os.path.join(DATA_DIR, "genes.txt"), "w") as f:
    for gene in shared_genes:
        f.write(gene + "\n")

samples.to_csv(os.path.join(DATA_DIR, "samples.csv"), index=False)

print("Saved spot_table.csv, genes.txt and samples.csv to", DATA_DIR)

## Step 6. Check the result

Quick checks before moving on:
- every section has spots, and the counts match the metadata roughly
- each patient's sections are listed together
- one section plotted on the slide, coloured by total expression, should look like tissue (not random noise). If it looks random, the barcode matching went wrong.

In [ ]:
summary = spot_table.groupby(["patient", "sample_id"]).size().rename("spots").reset_index()
summary = summary.merge(samples[["id", "spots_under_tissue"]], left_on="sample_id", right_on="id")
summary = summary.drop(columns="id").rename(columns={"spots_under_tissue": "spots_in_metadata"})
print(summary.to_string(index=False))
print()
print("Patients:", spot_table["patient"].nunique())
print("Sections:", spot_table["sample_id"].nunique())
print("Spots:", len(spot_table))
print("Genes:", len(shared_genes))

In [ ]:
check_id = sample_ids[0]
check_spots = spot_table[spot_table["sample_id"] == check_id]
check_expression = scipy.sparse.load_npz(os.path.join(EXPRESSION_DIR, f"{check_id}.npz"))
total_log_expression = np.asarray(check_expression.sum(axis=1)).ravel()

plt.figure(figsize=(5, 5))
plt.scatter(check_spots["x"], check_spots["y"], c=total_log_expression, s=20, cmap="viridis")
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.colorbar(label="sum of log1p expression")
plt.title(f"{check_id}: spots on the slide")
plt.show()

## Next

Run **Notebook 2 (Embedding)** in Colab on a **GPU** runtime. It reads `spot_table.csv` and the patch files, and saves one embedding file per section in the same spot order.

## Optional: filtered download and clean PNG/CSV export
The earlier filtered-download workflow queries HEST categories, lets you edit the selected IDs, then uses shared code in `src/hest_download_pipeline.py` to stage HDF5 files and export per-spot PNG, expression CSV, and barcode CSV files. This is an alternate clean-data layout for experiments; it is not required by Notebooks 02?03 and does not replace the canonical `raw/`, `spot_table.csv`, or `expression/` outputs above. Start with one sample because the download can be large.

In [ ]:
# This optional cell uses the project helper and its requirements.txt dependencies.
RUN_CLEAN_EXPORT = False
if RUN_CLEAN_EXPORT:
    import sys, subprocess
    from pathlib import Path
    here = Path.cwd().resolve()
    project_root = next((q for q in (here, *here.parents) if (q/'requirements.txt').is_file()), None)
    if project_root is None: raise RuntimeError('Run inside this project to use src/ helpers.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(project_root/'requirements.txt')])
    sys.path.insert(0, str(project_root/'src'))
    from hest_download_pipeline import download_clean_selection
    clean_dir = Path(DATA_DIR) / 'clean_export_preview'
    report = download_clean_selection(sample_ids[:1], output_dir=clean_dir, max_workers=1,
                                      metadata_filename=METADATA_FILE,
                                      staging_dir='/content' if Path('/content').is_dir() else None)
    display(report)
    print('Clean PNG/CSV export:', clean_dir)
else:
    print('Skipped. Set RUN_CLEAN_EXPORT=True to download/export one sample.')
